# Paper 10 · Attention Is All You Need

**Citation:** Ashish Vaswani et al., “Attention Is All You Need” (2017).

**Paper:** https://arxiv.org/abs/1706.03762

> **Scale gap:** We use a tiny Transformer on a synthetic order-sensitive classification task and inspect attention math directly.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 11 · Attention & Transformer Mathematics](../../math/11_attention_transformers.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. Why divide dot products by sqrt(d_k)?
2. Why does self-attention need positional information?
3. How does attention create direct paths between distant positions?

## Central claim
Sequence modeling can be performed with attention and feed-forward layers, removing recurrence and enabling more parallel computation.

## Scaled dot-product attention from scratch

In [ ]:
import math, numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
def attention(Q,K,V,mask=None):
    scores=Q@K.transpose(-2,-1)/math.sqrt(Q.shape[-1])
    if mask is not None: scores=scores.masked_fill(~mask,float("-inf"))
    w=torch.softmax(scores,-1)
    return w@V,w
Q=torch.randn(1,1,5,8); K=torch.randn(1,1,5,8); V=torch.randn(1,1,5,4)
out,w=attention(Q,K,V)
print("attention weights",w.shape,"row sums",w.sum(-1))

## Reproduce the positional-information requirement
Label depends on the **first and last** token. A bag/set of tokens is insufficient.

In [ ]:
torch.manual_seed(0)
N,T,VOC=2000,8,4
seq=torch.randint(0,VOC,(N,T))
label=((seq[:,0]==seq[:,-1])).long()
perm=torch.randperm(N); tr=perm[:1500]; te=perm[1500:]

class TinyTransformer(nn.Module):
    def __init__(self,use_pos=True):
        super().__init__(); D=24
        self.emb=nn.Embedding(VOC,D); self.use_pos=use_pos
        self.pos=nn.Parameter(torch.randn(T,D)*.02)
        layer=nn.TransformerEncoderLayer(D,4,48,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,2); self.head=nn.Linear(D,2)
    def forward(self,x):
        h=self.emb(x)
        if self.use_pos: h=h+self.pos
        h=self.enc(h)
        return self.head(h.mean(1))

def train(use_pos):
    torch.manual_seed(0); m=TinyTransformer(use_pos); opt=torch.optim.Adam(m.parameters(),lr=.005); ce=nn.CrossEntropyLoss(); hist=[]
    for _ in range(80):
        opt.zero_grad(); loss=ce(m(seq[tr]),label[tr]); loss.backward(); opt.step()
        with torch.no_grad(): acc=(m(seq[te]).argmax(1)==label[te]).float().mean().item()
        hist.append(acc)
    return np.array(hist)
hp=train(True); hn=train(False)
print("with position",hp[-1],"without",hn[-1])
plt.plot(hp,label="with position"); plt.plot(hn,label="no position"); plt.legend(); plt.ylabel("test accuracy"); plt.show()

### Scaling ablation
Generate random Q/K at increasing head dimension. Compare softmax entropy with and without the `1/sqrt(d_k)` factor.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?